# 01 - Data Cleaning & EDA : Online Retail II
Goal: load the row transactions, inspect data quality issue, document every cleaning decision with row counts, and save a clean dataset for RFM, market basket and recomendation notebooks.

In [6]:
import pandas as pd
import numpy as np
from pathlib  import Path

pd.set_option("display.max_columns", 50)

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_XLSX = RAW_DIR / "online_retail_II.xlsx"
RAW_PARQUET = RAW_DIR / "online_retail_raw.parquet"

print(PROJECT_DIR)
print("Excel found:", RAW_XLSX.exists())

C:\Users\user\Desktop\market-basket-next-product
Excel found: True


In [7]:
%%time
sheets = pd.read_excel(RAW_XLSX, sheet_name=None) # dict: {sheet name: DataFrame}

frames = []
for name, sheet_df in sheets.items():
    print(f"{name}: {sheet_df.shape}")
    frames.append(sheet_df.assign(sheet=name))   # add column with sheet name

df = pd.concat(frames, ignore_index = True)    # stack sheets vertically
print("Combined:", df.shape)
df.head()

Year 2009-2010: (525461, 8)
Year 2010-2011: (541910, 8)
Combined: (1067371, 9)
CPU times: total: 3min 57s
Wall time: 3min 58s


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [9]:
df = df.astype({
    "Invoice":     "string",
    "StockCode":   "string",
    "Description": "string",
    "Country":     "string",
    "sheet":       "string",
    "Customer ID": "Int64",
})
print(df.dtypes)

df.to_parquet(RAW_PARQUET, index=False)
print("Saved:", RAW_PARQUET.name, f"{RAW_PARQUET.stat().st_size / 1e6:.1f} MB")

Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                string
sheet                  string
dtype: object
Saved: online_retail_raw.parquet 7.2 MB


In [10]:
%%time
df = pd.read_parquet(RAW_PARQUET)
print(df.shape)
df.dtypes

(1067371, 9)
CPU times: total: 500 ms
Wall time: 498 ms


Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                string
sheet                  string
dtype: object

In [12]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct":     (df.isna().mean() * 100).round(2),
})
missing

,missing,pct
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00
sheet,0,0.00


In [15]:
print(df.groupby("sheet")["InvoiceDate"].agg(["min", "max", "count"]))

start_2nd = df.loc[df["sheet"] == "Year 2010-2011", "InvoiceDate"].min()
overlap = df[(df["sheet"] == "Year 2009-2010") & (df["InvoiceDate"] >= start_2nd)]
print("\nRows in sheet 1 dated after sheet 2 starts:", len(overlap))

                               min                 max   count
sheet                                                         
Year 2009-2010 2009-12-01 07:45:00 2010-12-09 20:01:00  525461
Year 2010-2011 2010-12-01 08:26:00 2011-12-09 12:50:00  541910

Rows in sheet 1 dated after sheet 2 starts: 22523


In [16]:
print("Full duplicates (incl. sheet column):", df.duplicated().sum())
print("Duplicates ignoring sheet column:   ", df.drop(columns="sheet").duplicated().sum())

Full duplicates (incl. sheet column): 12133
Duplicates ignoring sheet column:    34335


In [17]:
in_overlap = (df["sheet"] == "Year 2009-2010") & (df["InvoiceDate"] >= start_2nd)
df_no_overlap = df[~in_overlap]

print("Rows after removing overlap:", len(df_no_overlap))
print("Duplicates left:", df_no_overlap.drop(columns="sheet").duplicated().sum())

Rows after removing overlap: 1044848
Duplicates left: 11812


In [18]:
is_cancel = df["Invoice"].str.startswith("C")
neg_qty   = df["Quantity"] < 0
bad_price = df["Price"] <= 0

print("Cancellation invoices (C...):", is_cancel.sum())
print("Quantity < 0:               ", neg_qty.sum())
print("Price <= 0:                 ", bad_price.sum())

print("\nCancellation vs negative quantity:")
print(pd.crosstab(is_cancel, neg_qty, rownames=["cancel"], colnames=["qty<0"]))

print("\nInvoice prefixes that are not plain numbers:")
print(df.loc[~df["Invoice"].str.isdigit(), "Invoice"].str[0].value_counts())

print("\nShare of Price<=0 rows with missing Description:",
      round(df.loc[bad_price, "Description"].isna().mean(), 3))

Cancellation invoices (C...): 19494
Quantity < 0:                22950
Price <= 0:                  6207

Cancellation vs negative quantity:
qty<0     False  True 
cancel                
False   1044420   3457
True          1  19493

Invoice prefixes that are not plain numbers:
Invoice
C    19494
A        6
Name: count, dtype: int64[pyarrow]

Share of Price<=0 rows with missing Description: 0.706


In [19]:
df[neg_qty & ~is_cancel].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,sheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,<NA>,United Kingdom,Year 2009-2010
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom,Year 2009-2010
3114,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom,Year 2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,<NA>,United Kingdom,Year 2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,<NA>,United Kingdom,Year 2009-2010
4296,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,<NA>,United Kingdom,Year 2009-2010
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,<NA>,United Kingdom,Year 2009-2010
4566,489821,85049G,<NA>,-240,2009-12-02 13:25:00,0.0,<NA>,United Kingdom,Year 2009-2010


In [20]:
is_std = df["StockCode"].str.match(r"^\d{5}[A-Za-z]*$")
print("Rows with non-standard StockCode:", (~is_std).sum(),
      "| unique codes:", df.loc[~is_std, "StockCode"].nunique())

(df[~is_std]
 .groupby("StockCode")
 .agg(rows=("Invoice", "size"),
      example=("Description", "first"),
      avg_price=("Price", "mean"))
 .sort_values("rows", ascending=False)
 .head(25))

Rows with non-standard StockCode: 6094 | unique codes: 63


,rows,example,avg_price
StockCode,,,
POST,2122,POSTAGE,32.623322
DOT,1446,DOTCOM POSTAGE,223.144876
M,1421,Manual,528.776284
C2,282,CARRIAGE,49.460993
D,177,Discount,72.867288
S,104,SAMPLES,60.956731
BANK CHARGES,102,Bank Charges,359.034422
ADJUST,67,Adjustment by john on 26/01/2010 16,163.591343
AMAZONFEE,43,AMAZON FEE,7646.089302


In [21]:
print("Unique StockCodes:  ", df["StockCode"].nunique())
print("Unique Descriptions:", df["Description"].nunique())

desc_per_code = df.groupby("StockCode")["Description"].nunique()
print("Codes with >1 description:", (desc_per_code > 1).sum())

worst = desc_per_code.idxmax()
print(f"\nExample — {worst}:")
print(df.loc[df["StockCode"] == worst, "Description"].value_counts())

Unique StockCodes:   5305
Unique Descriptions: 5698
Codes with >1 description: 1232

Example — 20713:
Description
JUMBO BAG OWLS                  1372
missing                            1
wrongly marked. 23343 in box       1
wrongly coded-23343                1
found                              1
Found                              1
wrongly marked 23343               1
Marked as 23343                    1
wrongly coded 23343                1
Name: count, dtype: int64[pyarrow]


In [22]:
has_lower = df["Description"].str.contains(r"[a-z]", na=False)
print("Descriptions with lowercase letters:", has_lower.sum())
df.loc[has_lower, "Description"].value_counts().head(30)

Descriptions with lowercase letters: 5967


Description
Manual                                 1426
BAG 250g SWIRLY MARBLES                 598
BAG 125g SWIRLY MARBLES                 596
POLYESTER FILLER PAD 40x40cm            398
POLYESTER FILLER PAD 45x45cm            372
3 TRADITIONAl BISCUIT CUTTERS  SET      268
BAG 500g SWIRLY MARBLES                 240
FRENCH BLUE METAL DOOR SIGN No          186
Discount                                177
check                                   162
Bank Charges                             96
damages                                  84
damaged                                  81
ESSENTIAL BALM 3.5g TIN IN ENVELOPE      80
Next Day Carriage                        80
POLYESTER FILLER PAD 45x30cm             65
POLYESTER FILLER PAD 65CMx65CM           52
POLYESTER FILLER PAD 30CMx30CM           52
FRENCH BLUE METAL DOOR SIGN, No          48
NUMBER TILE COTTAGE GARDEN No            42
NUMBER TILE VINTAGE FONT No              39
Adjustment by john on 26/01/2010 16      38
FOLK ART GREETING CA